# Adaptive Fusion TS2Img — Colab command-only runner

Notebook này dùng đúng workflow nghiên cứu:

- **GitHub**: chứa source code.
- **Google Drive**: lưu outputs, cache, checkpoints, bảng kết quả và paper package.
- Notebook chỉ gọi lệnh; logic nằm trong `src/`; cấu hình nằm trong `config/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Clone source code từ GitHub

In [ ]:
from pathlib import Path
import os

GITHUB_REPO_URL = "https://github.com/hoangtnedu/adaptive-fusion-ts2img.git"
PROJECT_DIR = Path("/content/adaptive-fusion-ts2img")
DRIVE_ROOT = Path("/content/drive/MyDrive/research_ts2img_adaptive_fusion")

# Đặt True khi muốn lấy source mới nhất từ GitHub.
# Đặt False nếu muốn giữ lại thư mục /content hiện có trong cùng phiên Colab.
REFRESH_REPO = True

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "cache").mkdir(parents=True, exist_ok=True)

if REFRESH_REPO and PROJECT_DIR.exists():
    print("Removing old source folder in /content...")
    !rm -rf /content/adaptive-fusion-ts2img

if not PROJECT_DIR.exists():
    print("Cloning source code from GitHub...")
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}
else:
    print("Project already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}

print("Current project directory:")
!pwd
print("Current Git commit:")
!git rev-parse --short HEAD
print("Drive root:", DRIVE_ROOT)
print("Outputs:", DRIVE_ROOT / "outputs")
print("Cache:", DRIVE_ROOT / "cache")

## 2. Cài thư viện và kiểm tra source

In [ ]:
!pip install -r requirements.txt
!python -m compileall src

## 3. Tải dữ liệu UCR/UEA theo từng stage

In [ ]:
# Stage 1 datasets
!python -m src.cli.download_data \
  --config config/suites/stage1_smoke_5datasets_3seeds_all_methods.yaml \
  --data-root data/UCR

In [ ]:
# Stage 2 datasets
!python -m src.cli.download_data \
  --config config/suites/stage2_pilot_12datasets_3seeds_all_methods.yaml \
  --data-root data/UCR

In [ ]:
# Stage 3 datasets
!python -m src.cli.download_data \
  --config config/suites/stage3_paper_20datasets_3seeds_all_methods.yaml \
  --data-root data/UCR

## 4. Dry-run trước khi train

In [ ]:
# Kiểm tra số lệnh sẽ chạy, chưa train thật
!python -m src.cli.batch_run \
  --suite config/suites/stage1_smoke_5datasets_3seeds_all_methods.yaml \
  --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache \
  --dry-run

## 5. Stage 1 — Smoke test

In [ ]:
# Stage 1: 5 datasets x 3 seeds x 7 methods = 105 runs
!python -m src.cli.batch_run \
  --suite config/suites/stage1_smoke_5datasets_3seeds_all_methods.yaml \
  --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

## 6. Gom kết quả nhanh sau Stage 1

In [ ]:
!python -m src.cli.make_paper_package \
  --output-root {DRIVE_ROOT}/outputs \
  --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv \
  --out-dir {DRIVE_ROOT}/outputs/paper_package \
  --metric test_macro_f1 \
  --proposed adaptive_fusion_full

## 7. Stage 2 — Pilot experiment

In [ ]:
# Stage 2: 12 datasets x 3 seeds x 7 methods = 252 runs
!python -m src.cli.batch_run \
  --suite config/suites/stage2_pilot_12datasets_3seeds_all_methods.yaml \
  --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

## 8. Phân tích sau Stage 2

In [ ]:
!python -m src.cli.make_paper_package \
  --output-root {DRIVE_ROOT}/outputs \
  --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv \
  --out-dir {DRIVE_ROOT}/outputs/paper_package \
  --metric test_macro_f1 \
  --proposed adaptive_fusion_full

## 9. Stage 3 — Paper-grade minimum

In [ ]:
# Stage 3: 20 datasets x 3 seeds x 7 methods = 420 runs
!python -m src.cli.batch_run \
  --suite config/suites/stage3_paper_20datasets_3seeds_all_methods.yaml \
  --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

## 10. Ablation study cho bài báo

In [ ]:
# Ablation: full model và các cấu hình bỏ từng nhánh GAF/MTF/RP/STFT
!python -m src.cli.batch_run \
  --suite config/suites/stage3_ablation_20datasets_3seeds_adaptive.yaml \
  --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

## 11. Paper package cuối cùng

In [ ]:
!python -m src.cli.make_paper_package \
  --output-root {DRIVE_ROOT}/outputs \
  --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv \
  --out-dir {DRIVE_ROOT}/outputs/paper_package \
  --metric test_macro_f1 \
  --proposed adaptive_fusion_full

print("Paper package saved to:", DRIVE_ROOT / "outputs" / "paper_package")

## 12. Stage 4 — mở rộng nếu còn tài nguyên

In [ ]:
# Option A: 30 datasets x 5 seeds x 7 methods = 1050 runs
# !python -m src.cli.batch_run \
#   --suite config/suites/stage4_strong_30datasets_5seeds_all_methods.yaml \
#   --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

# Option B: 50 datasets x 3 seeds, chỉ chạy mô hình đề xuất
# !python -m src.cli.batch_run \
#   --suite config/suites/stage4_strong_50datasets_3seeds_adaptive_only.yaml \
#   --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache